# Red Siamesa (Triplet Loss) — Clasificación de hojas

Este notebook implementa el escenario experimental basado en una **red siamesa entrenada
con triplet loss**, sobre el mismo dataset de hojas (`train / valid / test`) usado en
`clasificacion_hojas_resnet_vs_vit.ipynb` y en `clasificacion_hojas_siamese_contrastive.ipynb`.

**Protocolo (según la consigna del curso):**
1. Entrenar una red siamesa con **triplet loss** para separar las muestras por clase: para
   cada ancla (anchor) se usa una imagen de la **misma clase** (positive) y una de **otra
   clase** (negative), y se entrena para que `d(anchor, positive) < d(anchor, negative)` con
   un margen mínimo.
2. **Tarea A:** congelar el backbone entrenado, añadir una capa **FC** y entrenar esa capa
   para clasificar las clases del dataset.
3. **Tarea B:** usar el mismo backbone congelado para extraer **vectores característicos**
   (embeddings) de cada imagen, y entrenar un modelo de **ML (XGBoost)** sobre esos vectores
   para clasificar las categorías.

La arquitectura del backbone (ResNet-50 + cabeza de proyección a `EMBED_DIM`) y el margen son
los mismos que en el notebook contrastive, para que ambos escenarios sean directamente
comparables en el informe final.


## 0. Dependencias

Si te falta algún paquete, descomenta y ejecuta la siguiente celda.

In [ ]:
!pip install torch torchvision scikit-learn matplotlib seaborn pandas tqdm xgboost


## 1. Imports y configuración

In [ ]:
import os
import time
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
)
from sklearn.decomposition import PCA

import xgboost as xgb

# tqdm.auto detecta automáticamente el entorno (notebook o consola)
# y no requiere ipywidgets funcionando para caer a la versión de texto
from tqdm.auto import tqdm


In [ ]:
# ----------------------------- CONFIGURACIÓN -----------------------------
DATA_DIR = Path("Dataset_split_Aug")          # debe contener train/ valid/ test/

IMG_SIZE     = 224
BATCH_SIZE   = 32                   # baja a 16 si te quedas sin memoria de GPU
NUM_WORKERS  = 0 if os.name == "nt" else 4   # 0 en Windows (evita bloqueos de multiprocessing en Jupyter)
SEED = 42

# --- Red siamesa (triplet loss) ---
EMBED_DIM         = 128             # dimensión del espacio de embeddings (igual que en contrastive)
TRIPLET_MARGIN    = 1.0             # margen del triplet loss (mismo valor que MARGIN en contrastive, para comparar)
EPOCHS_TRIPLET    = 10
LR_TRIPLET        = 3e-4
WEIGHT_DECAY      = 1e-4
TRIPLETS_PER_EPOCH_TRAIN = None     # se define tras cargar los datasets (2x tamaño de train)
TRIPLETS_VALID_FIXED     = 2000     # tripletas fijas de validación (reproducibles)

# --- Cabeza FC sobre backbone congelado (Tarea A) ---
EPOCHS_CLF = 10
LR_CLF     = 1e-3
LABEL_SMOOTHING = 0.1

OUTPUT_DIR = Path("checkpoints")
OUTPUT_DIR.mkdir(exist_ok=True)

# Reproducibilidad
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# Con tamaño de imagen fijo, benchmark=True deja que cuDNN elija los kernels más rápidos
torch.backends.cudnn.benchmark = torch.cuda.is_available()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = (DEVICE == "cuda")
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Transforms y Datasets

Usamos los mismos `train / valid / test` que los notebooks anteriores. `train_ds_eval` (sin
aumentos) se usa para extraer embeddings estables en la Tarea B.

In [ ]:
# Normalización estándar de ImageNet (los pesos preentrenados la esperan)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds      = datasets.ImageFolder(DATA_DIR / "train", transform=train_tf)   # con aumentos (para entrenar)
train_ds_eval = datasets.ImageFolder(DATA_DIR / "train", transform=eval_tf)    # sin aumentos (para extraer embeddings)
valid_ds      = datasets.ImageFolder(DATA_DIR / "valid", transform=eval_tf)
test_ds       = datasets.ImageFolder(DATA_DIR / "test",  transform=eval_tf)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)

pin = (DEVICE == "cuda")
# kwargs comunes: persistent_workers evita re-crear procesos en cada época (solo aplica si hay workers)
loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=pin,
                 persistent_workers=(NUM_WORKERS > 0))
train_loader      = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                               **loader_kw)
train_eval_loader = DataLoader(train_ds_eval, batch_size=BATCH_SIZE, shuffle=False,
                               **loader_kw)
valid_loader      = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False,
                               **loader_kw)
test_loader       = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                               **loader_kw)

print(f"Clases ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Imágenes -> train: {len(train_ds)} | valid: {len(valid_ds)} | test: {len(test_ds)}")


## 3. Dataset de tripletas (para triplet loss)

Cada muestra es una **tripleta** `(anchor, positive, negative)`: `anchor` y `positive` son
imágenes de la **misma clase** (elegidas al azar, distintas entre sí), y `negative` es una
imagen de una **clase distinta**. Para `valid` generamos un conjunto de tripletas **fijo**
(semilla fija) para que la validación sea reproducible entre épocas.

In [ ]:
class TripletDataset(Dataset):
    """Genera tripletas (anchor, positive, negative) a partir de un ImageFolder.
    anchor/positive -> misma clase (índices distintos)
    negative         -> clase distinta
    """
    def __init__(self, base_dataset, n_triplets, fixed=False, seed=None):
        self.base = base_dataset
        self.targets = np.array(base_dataset.targets)
        self.classes = np.unique(self.targets)
        self.class_to_indices = {c: np.where(self.targets == c)[0] for c in self.classes}
        self.n_triplets = n_triplets
        self.fixed = fixed
        if fixed:
            rng = np.random.RandomState(seed)
            self.triplets = [self._sample_triplet(rng) for _ in range(n_triplets)]

    def _sample_triplet(self, rng):
        idx_a = int(rng.randint(len(self.base)))
        y_a = self.targets[idx_a]
        same_class_idx = self.class_to_indices[y_a]
        # positive: misma clase, distinto índice al anchor (si la clase tiene >1 imagen)
        if len(same_class_idx) > 1:
            idx_p = idx_a
            while idx_p == idx_a:
                idx_p = int(rng.choice(same_class_idx))
        else:
            idx_p = idx_a
        # negative: clase distinta
        other_classes = self.classes[self.classes != y_a]
        y_n = rng.choice(other_classes)
        idx_n = int(rng.choice(self.class_to_indices[y_n]))
        return idx_a, idx_p, idx_n

    def __len__(self):
        return self.n_triplets

    def __getitem__(self, i):
        if self.fixed:
            idx_a, idx_p, idx_n = self.triplets[i]
        else:
            idx_a, idx_p, idx_n = self._sample_triplet(np.random)
        img_a, _ = self.base[idx_a]
        img_p, _ = self.base[idx_p]
        img_n, _ = self.base[idx_n]
        return img_a, img_p, img_n


TRIPLETS_PER_EPOCH_TRAIN = len(train_ds) * 2

train_triplet_ds = TripletDataset(train_ds, n_triplets=TRIPLETS_PER_EPOCH_TRAIN, fixed=False)
valid_triplet_ds = TripletDataset(valid_ds, n_triplets=TRIPLETS_VALID_FIXED, fixed=True, seed=SEED)

train_triplet_loader = DataLoader(train_triplet_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  **loader_kw)
valid_triplet_loader = DataLoader(valid_triplet_ds, batch_size=BATCH_SIZE, shuffle=False,
                                  **loader_kw)

print(f"Tripletas de entrenamiento por época: {len(train_triplet_ds)} | Tripletas de validación (fijas): {len(valid_triplet_ds)}")


## 4. Arquitectura de la red siamesa

Misma arquitectura que en el notebook contrastive: backbone **ResNet-50** preentrenado en
ImageNet + cabeza de proyección a un espacio de embeddings de dimensión `EMBED_DIM`,
normalizado con L2. Aquí se entrena una instancia **independiente** (pesos propios) usando
triplet loss.

In [ ]:
class SiameseNetwork(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        backbone = models.resnet50(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()      # quitamos la cabeza de clasificación original
        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, embed_dim),
        )

    def forward_once(self, x):
        feat = self.backbone(x)
        emb = self.embedding(feat)
        emb = F.normalize(emb, p=2, dim=1)   # embeddings en la hiperesfera unitaria
        return emb

    def forward(self, xa, xp, xn):
        return self.forward_once(xa), self.forward_once(xp), self.forward_once(xn)


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## 5. Triplet Loss

$$L = \max\big(0,\; d(a,p) - d(a,n) + m\big)$$

donde $d(a,p)$ es la distancia (euclidiana) entre anchor y positive, $d(a,n)$ entre anchor y
negative, y $m$ el margen. Se penaliza cuando el negativo no está al menos `margin` más
lejos que el positivo.

In [ ]:
class TripletLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, emb_a, emb_p, emb_n):
        d_ap = F.pairwise_distance(emb_a, emb_p)
        d_an = F.pairwise_distance(emb_a, emb_n)
        loss = torch.clamp(d_ap - d_an + self.margin, min=0)
        return loss.mean(), d_ap, d_an


## 6. Entrenamiento de la red siamesa

In [ ]:
def train_triplet(model, epochs=EPOCHS_TRIPLET, lr=LR_TRIPLET, margin=TRIPLET_MARGIN, name="siamese_triplet"):
    model = model.to(DEVICE)
    criterion = TripletLoss(margin=margin)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

    history = {"train_loss": [], "val_loss": [], "val_d_ap": [], "val_d_an": [],
               "train_active_frac": [], "val_triplet_acc": []}
    best_val_loss, best_state = float("inf"), None
    ckpt_path = OUTPUT_DIR / f"best_{name}.pt"

    print(f"\n===== Entrenando {name} | params entrenables: {count_trainable(model):,} =====")
    for epoch in range(1, epochs + 1):
        # ---------------- Train ----------------
        model.train()
        run_loss, seen, active = 0.0, 0, 0
        pbar = tqdm(train_triplet_loader, desc=f"[{name}] Época {epoch}/{epochs}", leave=False)
        for xa, xp, xn in pbar:
            xa = xa.to(DEVICE, non_blocking=True)
            xp = xp.to(DEVICE, non_blocking=True)
            xn = xn.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=USE_AMP):
                emb_a, emb_p, emb_n = model(xa, xp, xn)
                loss, d_ap, d_an = criterion(emb_a, emb_p, emb_n)
            # tripletas "activas": las que violan el margen y por tanto generan gradiente.
            # Si esta fracción cae cerca de 0, la red ya resolvió las tripletas fáciles
            # (señal de que convendría hard-negative mining o bajar el LR).
            active += (d_ap - d_an + margin > 0).sum().item()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)  # estabiliza el entrenamiento con AMP
            scaler.step(optimizer)
            scaler.update()

            run_loss += loss.item() * xa.size(0)
            seen += xa.size(0)
            pbar.set_postfix(loss=f"{run_loss/seen:.4f}", activas=f"{active/seen:.2%}")

        scheduler.step()
        train_loss = run_loss / seen
        train_active_frac = active / seen

        # ---------------- Validación ----------------
        model.eval()
        val_run_loss, val_seen, val_ok = 0.0, 0, 0
        val_d_ap_sum, val_d_an_sum = 0.0, 0.0
        with torch.no_grad():
            for xa, xp, xn in valid_triplet_loader:
                xa, xp, xn = xa.to(DEVICE), xp.to(DEVICE), xn.to(DEVICE)
                emb_a, emb_p, emb_n = model(xa, xp, xn)
                loss, d_ap, d_an = criterion(emb_a, emb_p, emb_n)
                val_run_loss += loss.item() * xa.size(0)
                val_d_ap_sum += d_ap.sum().item()
                val_d_an_sum += d_an.sum().item()
                val_ok += (d_ap < d_an).sum().item()   # tripleta bien ordenada
                val_seen += xa.size(0)
        val_loss = val_run_loss / val_seen
        val_d_ap = val_d_ap_sum / val_seen
        val_d_an = val_d_an_sum / val_seen
        val_triplet_acc = val_ok / val_seen

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_d_ap"].append(val_d_ap)
        history["val_d_an"].append(val_d_an)
        history["train_active_frac"].append(train_active_frac)
        history["val_triplet_acc"].append(val_triplet_acc)
        print(f"Época {epoch:2d} | train_loss {train_loss:.4f} (activas {train_active_frac:.2%}) "
              f"| val_loss {val_loss:.4f} | val d(a,p) {val_d_ap:.4f} | val d(a,n) {val_d_an:.4f} "
              f"| val triplet-acc {val_triplet_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, ckpt_path)

    print(f"Mejor val_loss ({name}): {best_val_loss:.4f}  ->  {ckpt_path}")
    model.load_state_dict(best_state)
    return model, history


In [ ]:
set_seed(SEED)
siamese_triplet = SiameseNetwork(embed_dim=EMBED_DIM)
t0 = time.time()
siamese_triplet, hist_triplet = train_triplet(siamese_triplet, epochs=EPOCHS_TRIPLET, lr=LR_TRIPLET, margin=TRIPLET_MARGIN)
time_triplet = time.time() - t0
print(f"Tiempo de entrenamiento red siamesa (triplet): {time_triplet/60:.1f} min")


### 6.1 Curva de pérdida y distancias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, len(hist_triplet["train_loss"]) + 1)

axes[0].plot(epochs_range, hist_triplet["train_loss"], "-o", label="train")
axes[0].plot(epochs_range, hist_triplet["val_loss"], "-o", label="valid")
axes[0].set_title("Triplet loss"); axes[0].set_xlabel("Época"); axes[0].legend()

axes[1].plot(epochs_range, hist_triplet["val_d_ap"], "-o", label="d(anchor, positive)", color="#2ca02c")
axes[1].plot(epochs_range, hist_triplet["val_d_an"], "-o", label="d(anchor, negative)", color="#d62728")
axes[1].set_title("Distancias promedio (validación)"); axes[1].set_xlabel("Época"); axes[1].legend()

plt.tight_layout(); plt.show()


### 6.2 Visualización de embeddings y separación de clases

Proyectamos los embeddings del set de validación a 2D con PCA, igual que en el notebook
contrastive, para comparar visualmente qué tan bien separa cada función de pérdida las
clases del dataset.

In [ ]:
@torch.no_grad()
def extract_embeddings(model, loader):
    model.eval()
    feats, labels = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        emb = model.forward_once(x)
        feats.append(emb.cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labels)


# --- Proyección PCA de embeddings de validación ---
X_valid_emb, y_valid_emb = extract_embeddings(siamese_triplet, valid_loader)
pca = PCA(n_components=2, random_state=SEED)
X_valid_2d = pca.fit_transform(X_valid_emb)

plt.figure(figsize=(7, 6))
scatter = plt.scatter(X_valid_2d[:, 0], X_valid_2d[:, 1], c=y_valid_emb, cmap="tab20", s=10)
plt.title("Embeddings de validación (PCA 2D) — red siamesa triplet")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.colorbar(scatter, label="Clase (índice)")
plt.tight_layout(); plt.show()

print(f"Distancia promedio final — d(a,p): {hist_triplet['val_d_ap'][-1]:.4f} "
      f"| d(a,n): {hist_triplet['val_d_an'][-1]:.4f}")


## 7. Tarea A — Clasificador FC sobre backbone congelado

Se congela el backbone de la red siamesa (pesos fijos, `eval()` permanente para las
capas `BatchNorm`) y se añade una única capa lineal (`FC`) que mapea el embedding a las
`NUM_CLASSES` categorías. Solo esta capa se entrena.

In [ ]:
class FrozenSiameseClassifier(nn.Module):
    def __init__(self, siamese_model, num_classes, embed_dim):
        super().__init__()
        self.siamese = siamese_model
        for p in self.siamese.parameters():
            p.requires_grad = False
        self.head = nn.Linear(embed_dim, num_classes)

    def train(self, mode=True):
        # El backbone siamés se mantiene siempre en eval() (BatchNorm congelado),
        # sin importar si el módulo completo está en modo train o eval.
        super().train(mode)
        self.siamese.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            emb = self.siamese.forward_once(x)
        return self.head(emb)


In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion=None):
    """Devuelve loss, accuracy, f1_macro y las predicciones/etiquetas completas."""
    model.eval()
    all_preds, all_labels = [], []
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        logits = model(x)
        if criterion is not None:
            total_loss += criterion(logits, y).item() * x.size(0)
        preds = logits.argmax(1)
        all_preds.append(preds.cpu()); all_labels.append(y.cpu())
        n += x.size(0)
    y_pred = torch.cat(all_preds).numpy()
    y_true = torch.cat(all_labels).numpy()
    loss = total_loss / n if criterion is not None else None
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro", zero_division=0)
    return loss, acc, f1m, y_true, y_pred


def train_model(model, name, epochs, lr):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(DEVICE, enabled=USE_AMP)

    history = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_f1"]}
    best_f1, best_state = -1.0, None
    ckpt_path = OUTPUT_DIR / f"best_{name}.pt"

    print(f"\n===== Entrenando {name} | params entrenables: {count_trainable(model):,} =====")
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss, run_correct, seen = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"[{name}] Época {epoch}/{epochs}", leave=False)
        for x, y in pbar:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=USE_AMP):
                logits = model(x)
                loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            run_loss += loss.item() * x.size(0)
            run_correct += (logits.argmax(1) == y).sum().item()
            seen += x.size(0)
            pbar.set_postfix(loss=f"{run_loss/seen:.3f}", acc=f"{run_correct/seen:.3f}")

        scheduler.step()
        train_loss = run_loss / seen
        train_acc = run_correct / seen

        val_loss, val_acc, val_f1, _, _ = evaluate(model, valid_loader, criterion)
        for k, v in zip(history, [train_loss, train_acc, val_loss, val_acc, val_f1]):
            history[k].append(v)
        print(f"Época {epoch:2d} | train_loss {train_loss:.3f} acc {train_acc:.3f} "
              f"| val_loss {val_loss:.3f} acc {val_acc:.3f} f1 {val_f1:.3f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, ckpt_path)

    print(f"Mejor F1-macro de validación ({name}): {best_f1:.4f}  ->  {ckpt_path}")
    model.load_state_dict(best_state)
    return model, history


In [ ]:
set_seed(SEED)
clf_fc_triplet = FrozenSiameseClassifier(siamese_triplet, NUM_CLASSES, EMBED_DIM)
t0 = time.time()
clf_fc_triplet, hist_fc_triplet = train_model(clf_fc_triplet, "siamese_triplet_fc", epochs=EPOCHS_CLF, lr=LR_CLF)
time_fc_triplet = time.time() - t0
print(f"Tiempo de entrenamiento cabeza FC: {time_fc_triplet/60:.1f} min")


### 7.1 Evaluación en test (Tarea A)

In [ ]:
def full_test_report(model, name, loader=test_loader):
    _, acc, f1m, y_true, y_pred = evaluate(model, loader)
    prec_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_m  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1_w   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n########## {name} — resultados en TEST ##########")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Precision (macro): {prec_m:.4f}")
    print(f"Recall    (macro): {rec_m:.4f}")
    print(f"F1        (macro): {f1m:.4f}")
    print(f"F1     (weighted): {f1_w:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    metrics = {"model": name, "accuracy": acc, "precision_macro": prec_m,
               "recall_macro": rec_m, "f1_macro": f1m, "f1_weighted": f1_w}
    return metrics, y_true, y_pred


metrics_fc_triplet, yt_fc_t, yp_fc_t = full_test_report(clf_fc_triplet, "Siamese-Triplet + FC")


## 8. Tarea B — Extracción de embeddings + XGBoost

Usamos el **mismo backbone siamés congelado** (sin la capa FC) para extraer un vector
característico por imagen, y entrenamos un `XGBClassifier` sobre esos vectores.

In [ ]:
X_train_emb_t, y_train_emb_t = extract_embeddings(siamese_triplet, train_eval_loader)
X_valid_emb_t, y_valid_emb_t = extract_embeddings(siamese_triplet, valid_loader)
X_test_emb_t,  y_test_emb_t  = extract_embeddings(siamese_triplet, test_loader)

print("Embeddings ->", "train:", X_train_emb_t.shape, "valid:", X_valid_emb_t.shape, "test:", X_test_emb_t.shape)


In [ ]:
xgb_clf_triplet = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softprob",
    num_class=NUM_CLASSES,
    eval_metric="mlogloss",
    random_state=SEED,
    n_jobs=-1,
    early_stopping_rounds=20,
)

t0 = time.time()
xgb_clf_triplet.fit(
    X_train_emb_t, y_train_emb_t,
    eval_set=[(X_valid_emb_t, y_valid_emb_t)],
    verbose=False,
)
time_xgb_triplet = time.time() - t0
print(f"Tiempo de entrenamiento XGBoost: {time_xgb_triplet:.1f} s | mejor iteración: {xgb_clf_triplet.best_iteration}")

xgb_clf_triplet.save_model(OUTPUT_DIR / "xgb_siamese_triplet.json")


### 8.1 Evaluación en test (Tarea B)

In [ ]:
def evaluate_predictions(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    prec_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec_m  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1m    = f1_score(y_true, y_pred, average="macro", zero_division=0)
    f1_w   = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    print(f"\n########## {name} — resultados en TEST ##########")
    print(f"Accuracy         : {acc:.4f}")
    print(f"Precision (macro): {prec_m:.4f}")
    print(f"Recall    (macro): {rec_m:.4f}")
    print(f"F1        (macro): {f1m:.4f}")
    print(f"F1     (weighted): {f1_w:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    metrics = {"model": name, "accuracy": acc, "precision_macro": prec_m,
               "recall_macro": rec_m, "f1_macro": f1m, "f1_weighted": f1_w}
    return metrics


y_pred_xgb_t = xgb_clf_triplet.predict(X_test_emb_t)
metrics_xgb_triplet = evaluate_predictions(y_test_emb_t, y_pred_xgb_t, "Siamese-Triplet + XGBoost")


## 9. Comparación de resultados

In [ ]:
summary = pd.DataFrame([metrics_fc_triplet, metrics_xgb_triplet]).set_index("model")
summary["train_time_min"] = [time_fc_triplet / 60, time_xgb_triplet / 60]
summary["trainable_params_M"] = [count_trainable(clf_fc_triplet) / 1e6, np.nan]

display(summary.round(4))

summary.round(4).to_csv(OUTPUT_DIR / "comparacion_metricas_siamese_triplet.csv")
print("Métricas guardadas en", OUTPUT_DIR / "comparacion_metricas_siamese_triplet.csv")

# Si ya corriste los notebooks anteriores, esto arma la tabla comparativa completa
# de los 6 escenarios pedidos en la consigna: CNN, ViT, siamese-contrastive (FC/XGB) y siamese-triplet (FC/XGB)
pieces = [summary]
for fname in ["comparacion_metricas.csv", "comparacion_metricas_siamese_contrastive.csv"]:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        prev = pd.read_csv(fpath, index_col=0)
        pieces.append(prev[summary.columns.intersection(prev.columns)])

if len(pieces) > 1:
    combined = pd.concat(pieces).drop_duplicates()
    print("\nTabla comparativa combinada de todos los escenarios disponibles:")
    display(combined.round(4))
    combined.round(4).to_csv(OUTPUT_DIR / "comparacion_metricas_TODOS_los_escenarios.csv")
    print("Guardada en", OUTPUT_DIR / "comparacion_metricas_TODOS_los_escenarios.csv")


In [ ]:
metric_cols = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]
ax = summary[metric_cols].T.plot(kind="bar", figsize=(11, 4.5), color=["#9467bd", "#8c564b"])
ax.set_title("Siamese-Triplet: FC-head vs. XGBoost — métricas en test")
ax.set_ylabel("Puntaje"); ax.set_ylim(0, 1)
ax.legend(title="Modelo"); plt.xticks(rotation=20, ha="right")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8, padding=2)
plt.tight_layout(); plt.show()


### 9.1 Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(2 * max(6, NUM_CLASSES * 0.7), max(5, NUM_CLASSES * 0.6)))
for ax, (yt, yp, title) in zip(
        axes,
        [(yt_fc_t, yp_fc_t, "Siamese-Triplet + FC"), (y_test_emb_t, y_pred_xgb_t, "Siamese-Triplet + XGBoost")]):
    cm = confusion_matrix(yt, yp)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Purples", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f"Matriz de confusión — {title}")
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
    ax.tick_params(axis="x", rotation=45); ax.tick_params(axis="y", rotation=0)
plt.tight_layout(); plt.show()


## 10. Conclusiones

Rellena esta sección con tus observaciones a partir de los resultados anteriores. Algunas
preguntas guía para tu informe:

- **¿El triplet loss logró separar las clases?** Revisa la sección 6.1/6.2: ¿`d(a,n)` se alejó
  claramente de `d(a,p)` a lo largo del entrenamiento? ¿Se ven agrupamientos por clase en la
  proyección PCA?
- **¿FC-head o XGBoost obtuvo mejor F1-macro / accuracy en test?**
- **Contrastive vs. triplet:** con la misma arquitectura, el mismo margen y el mismo número
  de épocas, ¿qué función de pérdida generó embeddings más separables (compara las secciones
  6.2 de ambos notebooks y la tabla combinada de la sección 9)?
- **¿Cómo se comparan los 6 escenarios en total** (CNN, ViT, siamese-contrastive×{FC, XGBoost},
  siamese-triplet×{FC, XGBoost})? Usa `checkpoints/comparacion_metricas_TODOS_los_escenarios.csv`
  (generado en la sección 9) como base para la sección de resultados y discusión del informe.
- **¿En qué clases falla más cada clasificador?** Compara las matrices de confusión entre
  escenarios: ¿son las mismas clases difíciles en todos los casos, o cambia según el método?
- Con esto se completan los 6 escenarios experimentales pedidos en la consigna: ya puedes
  redactar introducción, metodología, resultados/discusión y conclusiones del informe final.
